In [7]:
#%pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
import csv

print(np.__version__)

2.5.3


In [8]:
# Função para carregar dados do CSV
def carregar_dados(arquivo):
    X, y = [], []
    with open(arquivo, 'r') as f:
        leitor = csv.reader(f)
        next(leitor)  # pular cabeçalho
        for linha in leitor:
            X.append([float(linha[0]), float(linha[1])])  # idade, salario
            y.append(int(linha[2]))
    return np.array(X), np.array(y)

# Carregando o dataset
X, y = carregar_dados("dataset.csv")
print("Formato dos dados:", X.shape, y.shape)
print("Primeiras amostras:\n", X[:5], "\nRótulos:", y[:5])

Formato dos dados: (10, 2) (10,)
Primeiras amostras:
 [[  22. 2000.]
 [  25. 2200.]
 [  47. 4300.]
 [  52. 4600.]
 [  46. 4100.]] 
Rótulos: [0 0 1 1 1]


In [9]:
dataset = np.loadtxt("dataset.csv", delimiter=',', skiprows=1)

# Separa as colunas de características (features) e a coluna de rótulos (labels)

# X_train recebe todas as linhas, mas apenas as duas primeiras colunas (índice 0 e 1)
X_train = dataset[:, :2]

# y_train recebe todas as linhas, mas apenas a última coluna (índice 2)
# .astype(int) para garantir que as classes sejam tratadas como inteiros
y_train = dataset[:, 2].astype(int)

print("Amostra das características (X_train):")
print(X_train[:5])
print("\nAmostra dos rótulos (y_train):")
print(y_train[:5])

Amostra das características (X_train):
[[  22. 2000.]
 [  25. 2200.]
 [  47. 4300.]
 [  52. 4600.]
 [  46. 4100.]]

Amostra dos rótulos (y_train):
[0 0 1 1 1]


In [10]:
def entropia(y):
    classes, contagem = np.unique(y, return_counts=True)
    probs = contagem / len(y)
    return -np.sum(probs * np.log2(probs + 1e-9))

# Teste da entropia
print("Entropia [0,0,0,0] ->", entropia([0,0,0,0]))
print("Entropia [0,1,0,1] ->", entropia([0,1,0,1]))

Entropia [0,0,0,0] -> -1.4426951595367387e-09
Entropia [0,1,0,1] -> 0.99999999711461


In [ ]:
def dividir(X, y, feature, limiar):
    esquerda_idx = X[:, feature] <= limiar
    direita_idx = X[:, feature] > limiar
    return X[esquerda_idx], y[esquerda_idx], X[direita_idx], y[direita_idx]

# Testando divisão por idade <= 40
Xe, ye, Xd, yd = dividir(X, y, feature=0, limiar=40)
print("Esquerda:", ye)
print("Direita:", yd)

In [ ]:
class No:
    def __init__(self, feature=None, limiar=None, esquerda=None, direita=None, valor=None):
        self.feature = feature
        self.limiar = limiar
        self.esquerda = esquerda
        self.direita = direita
        self.valor = valor  # classe se for folha

class ArvoreDecisao:
    def __init__(self, max_profundidade=3, min_amostras=2):
        self.max_profundidade = max_profundidade
        self.min_amostras = min_amostras
        self.raiz = None

    def ajustar(self, X, y):
        self.raiz = self._crescer(X, y, profundidade=0)

    def _crescer(self, X, y, profundidade):
        num_amostras, num_features = X.shape
        num_classes = len(np.unique(y))        